### Setup

In [28]:
import os
import json
import random
import time
import requests
from bs4 import BeautifulSoup
import snowflake.connector
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database="RAW",
)

def run_query(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

HEADERS = {
    "User-Agent":      "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/122.0.0.0 Safari/537.36",
    "Accept":          "text/html,application/xhtml+xml,application/xml;"
                       "q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer":         "https://www.builtinnyc.com/",
}
MIN_SCRAPE_DELAY = 8.0
MAX_SCRAPE_DELAY = 15.0
KNOWN_SENIORITY_VALUES = {"Entry level", "Junior", "Mid level"}

print("Setup OK")

Setup OK


### Fetch candidates

In [29]:
candidates = run_query("""
    SELECT
        RAW_PAYLOAD:source_url::STRING as source_url,
        RAW_PAYLOAD:crawl_title::STRING as title
    FROM RAW.BUILTIN.SRC_POSTINGS
    WHERE RAW_PAYLOAD:seniority::STRING IS NULL
    ORDER BY INGESTED_AT DESC
""")

print(f"Candidates to backfill: {len(candidates)}")
display(candidates)

Candidates to backfill: 50


,SOURCE_URL,TITLE
0,https://www.builtinnyc.com/job/data-engineer-m...,Data Engineer - Manager
1,https://www.builtinnyc.com/job/fp-analyst-data...,FP&A Analyst - Data Insights
2,https://www.builtinnyc.com/job/data-analyst/96...,Data Analyst
3,https://www.builtinnyc.com/job/data-analyst-do...,"Data Analyst, DOC"
4,https://www.builtinnyc.com/job/data-analyst-qa...,Data Analyst/QA Engineer
5,https://www.builtinnyc.com/job/operations-data...,Operations Data Analyst- Insurance
6,https://www.builtinnyc.com/job/digital-data-an...,Digital Data Analyst
7,https://www.builtinnyc.com/job/marketing-and-c...,Marketing and Customer Data Analyst
8,https://www.builtinnyc.com/job/market-data-ana...,Market Data Analyst
9,https://www.builtinnyc.com/job/data-analyst/96...,Data Analyst


### Seniority Extraction Helper

In [30]:
def extract_seniority(url: str) -> str:
    try:
        delay = random.uniform(MIN_SCRAPE_DELAY, MAX_SCRAPE_DELAY)
        print(f"  sleeping {delay:.1f}s...")
        time.sleep(delay)
        
        response = requests.get(url, headers=HEADERS, timeout=15)
        print(f"  status: {response.status_code}  html_len: {len(response.text)}")

        if response.status_code != 200:
            return "unavailable"

        if len(response.text) < 50000:
            print("  WARNING: response suspiciously small, likely bot challenge")
            return "unavailable"

        soup = BeautifulSoup(response.text, "html.parser")

        for div in soup.find_all("div", class_=lambda c: c and "align-items-start" in c and "gap-sm" in c):
            if div.find("i", class_=lambda c: c and "fa-trophy" in c):
                span = div.find("span")
                if span:
                    text = span.get_text(strip=True)
                    if text in KNOWN_SENIORITY_VALUES:
                        return text
                    else:
                        print(f"  Unexpected value: {text!r}")
                        return "not_found"

        return "not_found"

    except Exception as e:
        print(f"  Error: {e}")
        return "unavailable"

print("Extractor OK")

Extractor OK


### Gate Check

In [31]:
# confirm unblocked and selector works before touching Snowflake
test_url = candidates.iloc[0]["SOURCE_URL"]
print(f"Gate check: {test_url}\n")

response = requests.get(test_url, headers=HEADERS, timeout=15)
print(f"status: {response.status_code}  html_len: {len(response.text)}")

soup = BeautifulSoup(response.text, "html.parser")
trophy = soup.find("i", class_=lambda c: c and "fa-trophy" in c)
print(f"Trophy found: {trophy is not None}")

if trophy:
    wrapper = trophy.parent.parent
    span = wrapper.find("span")
    print(f"Span text: {span.get_text(strip=True) if span else None}")

print()
if len(response.text) < 50000 or trophy is None:
    print("❌ GATE FAILED — do not proceed. Wait and retry.")
else:
    print("✅ GATE PASSED — safe to continue.")

Gate check: https://www.builtinnyc.com/job/data-engineer-manager/9657210

status: 200  html_len: 87531
Trophy found: True
Span text: Mid level

✅ GATE PASSED — safe to continue.


### Single Row Test

In [32]:
test_url = candidates.iloc[0]["SOURCE_URL"]
test_title = candidates.iloc[0]["TITLE"]

print(f"Testing: {test_title}")
print(f"URL: {test_url}\n")

seniority = extract_seniority(test_url)
print(f"\nExtracted seniority: {seniority!r}")

Testing: Data Engineer - Manager
URL: https://www.builtinnyc.com/job/data-engineer-manager/9657210

  sleeping 8.6s...
  status: 200  html_len: 87531

Extracted seniority: 'Mid level'


In [18]:
response = requests.get(test_url, headers=HEADERS, timeout=15)
print(f"  status: {response.status_code}  html_len: {len(response.text)}")

  status: 200  html_len: 16053


### Write Single Test Row to Snowflake

In [33]:
cur = conn.cursor()
cur.execute("""
    UPDATE RAW.BUILTIN.SRC_POSTINGS
    SET RAW_PAYLOAD = OBJECT_INSERT(RAW_PAYLOAD, 'seniority', %s, true)
    WHERE RAW_PAYLOAD:source_url::STRING = %s
""", (seniority, test_url))
conn.commit()
cur.close()
print(f"Updated 1 row — seniority set to {seniority!r}")

Updated 1 row — seniority set to 'Mid level'


### Verify the Single Test Row

In [34]:
verify = run_query(f"""
    SELECT
        RAW_PAYLOAD:source_url::STRING as source_url,
        RAW_PAYLOAD:crawl_title::STRING as title,
        RAW_PAYLOAD:seniority::STRING as seniority
    FROM RAW.BUILTIN.SRC_POSTINGS
    WHERE RAW_PAYLOAD:source_url::STRING = '{test_url}'
""")
display(verify)

,SOURCE_URL,TITLE,SENIORITY
0,https://www.builtinnyc.com/job/data-engineer-m...,Data Engineer - Manager,Mid level


### Dry Run Remaining Rows (No Wrties)

In [35]:
remaining = candidates[candidates["SOURCE_URL"] != test_url].reset_index(drop=True)

results = []
for i, row in remaining.iterrows():
    print(f"[{i+1}/{len(remaining)}] {row['TITLE'][:50]}")
    seniority = extract_seniority(row["SOURCE_URL"])

    if seniority == "unavailable" and i < 3:
        print("❌ Early unavailable — possible block. Stopping dry run.")
        break

    print(f"  → {seniority!r}")
    results.append({
        "source_url": row["SOURCE_URL"],
        "title": row["TITLE"],
        "seniority": seniority
    })

results_df = pd.DataFrame(results)
print("\nDry run complete")
display(results_df)
print(results_df["seniority"].value_counts())

[1/49] FP&A Analyst - Data Insights
  sleeping 12.6s...
  status: 200  html_len: 89088
  → 'Junior'
[2/49] Data Analyst
  sleeping 13.8s...
  status: 200  html_len: 88155
  → 'Junior'
[3/49] Data Analyst, DOC
  sleeping 8.4s...
  status: 200  html_len: 90746
  → 'Junior'
[4/49] Data Analyst/QA Engineer
  sleeping 14.1s...
  status: 200  html_len: 90109
  → 'Mid level'
[5/49] Operations Data Analyst- Insurance
  sleeping 8.4s...
  status: 200  html_len: 82494
  Unexpected value: 'Senior level'
  → 'not_found'
[6/49] Digital Data Analyst
  sleeping 14.1s...
  status: 200  html_len: 77241
  → 'Junior'
[7/49] Marketing and Customer Data Analyst
  sleeping 12.1s...
  status: 200  html_len: 78066
  → 'Mid level'
[8/49] Market Data Analyst
  sleeping 13.0s...
  status: 200  html_len: 86754
  → 'Entry level'
[9/49] Data Analyst
  sleeping 10.1s...
  status: 200  html_len: 79000
  Unexpected value: 'Senior level'
  → 'not_found'
[10/49] Systems Analyst - Data Scientist
  sleeping 8.5s...
  stat

,source_url,title,seniority
0,https://www.builtinnyc.com/job/fp-analyst-data...,FP&A Analyst - Data Insights,Junior
1,https://www.builtinnyc.com/job/data-analyst/96...,Data Analyst,Junior
2,https://www.builtinnyc.com/job/data-analyst-do...,"Data Analyst, DOC",Junior
3,https://www.builtinnyc.com/job/data-analyst-qa...,Data Analyst/QA Engineer,Mid level
4,https://www.builtinnyc.com/job/operations-data...,Operations Data Analyst- Insurance,not_found
5,https://www.builtinnyc.com/job/digital-data-an...,Digital Data Analyst,Junior
6,https://www.builtinnyc.com/job/marketing-and-c...,Marketing and Customer Data Analyst,Mid level
7,https://www.builtinnyc.com/job/market-data-ana...,Market Data Analyst,Entry level
8,https://www.builtinnyc.com/job/data-analyst/96...,Data Analyst,not_found
9,https://www.builtinnyc.com/job/systems-analyst...,Systems Analyst - Data Scientist,Junior


seniority
unavailable    22
Mid level      13
Junior          9
not_found       3
Entry level     2
Name: count, dtype: int64


In [36]:
from IPython.display import HTML

check_df = results_df.copy()
check_df["url"] = check_df["source_url"].apply(lambda x: f'<a href="{x}" target="_blank">{x}</a>')

HTML(check_df[["title", "seniority", "url"]].to_html(render_links=True, escape=False))

,title,seniority,url
0,FP&A Analyst - Data Insights,Junior,https://www.builtinnyc.com/job/fp-analyst-data-insights/9662277
1,Data Analyst,Junior,https://www.builtinnyc.com/job/data-analyst/9683737
2,"Data Analyst, DOC",Junior,https://www.builtinnyc.com/job/data-analyst-doc/9683670
3,Data Analyst/QA Engineer,Mid level,https://www.builtinnyc.com/job/data-analyst-qa-engineer/9693624
4,Operations Data Analyst- Insurance,not_found,https://www.builtinnyc.com/job/operations-data-analyst-insurance/9691977
5,Digital Data Analyst,Junior,https://www.builtinnyc.com/job/digital-data-analyst/9688896
6,Marketing and Customer Data Analyst,Mid level,https://www.builtinnyc.com/job/marketing-and-customer-data-analyst/9688892
7,Market Data Analyst,Entry level,https://www.builtinnyc.com/job/market-data-analyst/9656987
8,Data Analyst,not_found,https://www.builtinnyc.com/job/data-analyst/9640713
9,Systems Analyst - Data Scientist,Junior,https://www.builtinnyc.com/job/systems-analyst-data-scientist/9655527


In [37]:
from pathlib import Path

CACHE_DIR = Path("./cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
BACKFILL_CHECKPOINT = CACHE_DIR / "backfill_results_checkpoint.csv"

results_df.to_csv(BACKFILL_CHECKPOINT, index=False)
print(f"Saved {len(results_df)} rows to {BACKFILL_CHECKPOINT.resolve()}")

Saved 49 rows to /Users/vanbrantley/code/nyc-data-job-market-tracker/ingestion/notebooks/cache/backfill_results_checkpoint.csv


<!-- Got up to 30, 24 and 29 say a seniority but should be not_found. Pulled from Similar Job's seniority. -->

### Bulk Write After Dry Run Looks Good

In [ ]:
# cur = conn.cursor()
# updated = 0
# for _, row in results_df.iterrows():
#     cur.execute("""
#         UPDATE RAW.BUILTIN.SRC_POSTINGS
#         SET RAW_PAYLOAD = OBJECT_INSERT(RAW_PAYLOAD, 'seniority', %s, true)
#         WHERE RAW_PAYLOAD:source_url::STRING = %s
#     """, (row["seniority"], row["source_url"]))
#     updated += 1

# conn.commit()
# cur.close()
# print(f"Updated {updated} rows")

### Final Verification

In [ ]:
# final = run_query("""
#     SELECT
#         RAW_PAYLOAD:seniority::STRING as seniority,
#         COUNT(*) as cnt
#     FROM RAW.BUILTIN.SRC_POSTINGS
#     GROUP BY 1
#     ORDER BY cnt DESC
# """)
# display(final)